# 41 — Cross Reference Existing Outputs (string-date fallback)
Avoid tz-aware/tz-naive sort/groupby failures by converting all upstream dates to canonical `YYYY-MM-DD` keys before merging.

In [ ]:
import os, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {"python": sys.version, "platform": platform.platform()},
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def to_date_key_series(series):
    dt = pd.to_datetime(series, errors="coerce", utc=True)
    dt = dt.dt.tz_convert(None).dt.normalize()
    return dt.dt.strftime("%Y-%m-%d")

def add_date_key(df):
    out = df.copy()
    for c in out.columns:
        if str(c).lower() in {"date","day","datetime","timestamp","time","time_utc","publishedat","published_at"}:
            out["date_key"] = to_date_key_series(out[c])
            return out
    return out

cfg = load_yaml(CONFIGS / "run.yml")
run_cfg = cfg.get("run", {})
date_from = run_cfg.get("date_from", "2025-01-01")
date_to = run_cfg.get("date_to", "2025-01-31")
sites = cfg.get("sites", [])
sites_df = pd.DataFrame(sites)
if sites_df.empty:
    sites_df = pd.DataFrame(columns=["id","name","role","lat","lon"])

In [ ]:
step = "41_cross_reference_existing_outputs"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

registry_path = OUTPUTS / "40_target_control_extension" / "sites_registry.csv"
registry, registry_meta = safe_read_csv(registry_path)
if registry.empty:
    registry = sites_df.copy()

base_daily = pd.DataFrame({"date": pd.date_range(date_from, date_to, freq="D")})
base_daily["date_key"] = base_daily["date"].dt.strftime("%Y-%m-%d")

weather_candidates = [
    "outputs/05_metoffice_datahub_weather/weather_hourly.csv",
    "outputs/newhaven_fusion/weather_hourly.csv",
    "outputs/32_newhaven_weather_ground_news_fusion/weather_hourly.csv",
    "outputs/32_newhaven_fusion/weather_hourly.csv",
]
ground_candidates = [
    "outputs/04_ground_aq_providers/ground_daily.csv",
    "outputs/newhaven_fusion/ground_data_daily.csv",
    "outputs/32_newhaven_weather_ground_news_fusion/ground_data_daily.csv",
    "outputs/32_newhaven_fusion/ground_data_daily.csv",
]
news_candidates = [
    "outputs/03_news_context/news_articles.json",
    "outputs/newhaven_fusion/news_articles.csv",
    "outputs/32_newhaven_weather_ground_news_fusion/news_articles.csv",
    "outputs/32_newhaven_fusion/news_articles.csv",
]

weather = base_daily[["date_key"]].copy()
weather_meta = []
for cand in weather_candidates:
    df, meta = safe_read_csv(cand)
    weather_meta.append(meta)
    if not df.empty:
        df = add_date_key(df)
        if "date_key" not in df.columns:
            continue
        keep = {}
        for c in ["wind_speed_ms","wind_dir_deg","cloud_cover_pct","visibility_m","temperature_c",
                  "wind_speed_10m","wind_direction_10m","cloud_cover","visibility","temperature_2m"]:
            if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
                keep[c] = "mean"
        if keep:
            grouped = df.groupby("date_key", as_index=False, dropna=True).agg(keep)
            weather = base_daily[["date_key"]].merge(grouped, on="date_key", how="left")
            break

ground = base_daily[["date_key"]].copy()
ground_meta = []
for cand in ground_candidates:
    df, meta = safe_read_csv(cand)
    ground_meta.append(meta)
    if not df.empty:
        df = add_date_key(df)
        if "date_key" not in df.columns:
            continue
        numeric_cols = [c for c in df.columns if c != "date_key" and pd.api.types.is_numeric_dtype(df[c])]
        agg = {c: "mean" for c in numeric_cols[:8]}
        if agg:
            grouped = df.groupby("date_key", as_index=False, dropna=True).agg(agg)
            counts = df.groupby("date_key").size().rename("ground_rows").reset_index()
            ground = base_daily[["date_key"]].merge(grouped, on="date_key", how="left").merge(counts, on="date_key", how="left")
            break

news = base_daily[["date_key"]].copy()
news["article_count"] = 0
news["headline_sample"] = ""
news_meta = []
for cand in news_candidates:
    p = PROJECT_ROOT / cand
    if p.suffix.lower() == ".json" and p.exists():
        try:
            raw = load_json(p)
        except Exception as e:
            news_meta.append({"path": str(p), "status": f"json_error:{e}"})
            continue
        rows = []
        for a in raw:
            published = a.get("publishedAt") or a.get("published_at") or a.get("date")
            title = a.get("title", "")
            if published:
                key = to_date_key_series(pd.Series([published])).iloc[0]
                rows.append({"date_key": key, "title": title})
        if rows:
            ndf = pd.DataFrame(rows).dropna(subset=["date_key"])
            grouped = ndf.groupby("date_key", as_index=False, dropna=True).agg(
                article_count=("title","size"),
                headline_sample=("title", lambda x: " | ".join(list(map(str, list(x)[:3]))))
            )
            news = base_daily[["date_key"]].merge(grouped, on="date_key", how="left").fillna({"article_count": 0, "headline_sample": ""})
        news_meta.append({"path": str(p), "status": "ok", "sha256": sha256_file(p)})
        break
    else:
        df, meta = safe_read_csv(cand)
        news_meta.append(meta)
        if not df.empty:
            df = add_date_key(df)
            if "date_key" not in df.columns:
                continue
            title_col = next((c for c in df.columns if c.lower() in ["title","headline"]), None)
            if not title_col:
                df["title"] = ""
                title_col = "title"
            grouped = df.groupby("date_key", as_index=False, dropna=True).agg(
                article_count=(title_col,"size"),
                headline_sample=(title_col, lambda x: " | ".join(list(map(str, list(x)[:3]))))
            )
            news = base_daily[["date_key"]].merge(grouped, on="date_key", how="left").fillna({"article_count": 0, "headline_sample": ""})
            break

daily_global = base_daily.merge(weather, on="date_key", how="left").merge(ground, on="date_key", how="left").merge(news, on="date_key", how="left")

per_site = []
for _, site in registry.iterrows():
    t = daily_global.copy()
    t["site_id"] = site.get("id")
    t["site_name"] = site.get("name")
    t["role"] = site.get("role")
    t["lat"] = site.get("lat")
    t["lon"] = site.get("lon")
    t["site_score"] = pd.to_numeric(t.get("article_count", 0), errors="coerce").fillna(0)
    per_site.append(t)

xref = pd.concat(per_site, ignore_index=True) if per_site else daily_global.copy()
xref.to_csv(out / "cross_reference_daily.csv", index=False)

manifest = manifest_base(step, [CONFIGS / "run.yml"])
manifest["artifacts"].append({"path": str(out / "cross_reference_daily.csv"), "sha256": sha256_file(out / "cross_reference_daily.csv")})
manifest["inputs"] = {"registry": registry_meta, "weather": weather_meta, "ground": ground_meta, "news": news_meta}
write_json(out / "manifest.json", manifest)

display(xref.head(20))
print("Wrote:", out)